In [18]:
import pandas as pd

url = "https://raw.githubusercontent.com/YBI-Foundation/Dataset/main/Bank%20Churn%20Modelling.csv"
df = pd.read_csv(url)

# Inspect the shape and first few rows
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (10000, 13)


,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,Num Of Products,Has Credit Card,Is Active Member,Estimated Salary,Churn
0,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [19]:
print("columns:",df.columns)
print("info:",df.info())
print("describe:",df.describe())
print("missing values:",df.isnull().sum())
print("duplicates:",df.duplicated().sum())


columns: Index(['CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age',
       'Tenure', 'Balance', 'Num Of Products', 'Has Credit Card',
       'Is Active Member', 'Estimated Salary', 'Churn'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   CustomerId        10000 non-null  int64  
 1   Surname           10000 non-null  object 
 2   CreditScore       10000 non-null  int64  
 3   Geography         10000 non-null  object 
 4   Gender            10000 non-null  object 
 5   Age               10000 non-null  int64  
 6   Tenure            10000 non-null  int64  
 7   Balance           10000 non-null  float64
 8   Num Of Products   10000 non-null  int64  
 9   Has Credit Card   10000 non-null  int64  
 10  Is Active Member  10000 non-null  int64  
 11  Estimated Salary  10000 non-null  float64
 12

In [20]:
#noted that there are no missing values or duplicates
#no outlier values, all attributes are well within limits
#Customer Id and Surname have no mathematical contributinon to the model and hence, will be dropped
# geography and gender are of object datatype so we convert them into sklearn-readable vales using one hot encoding
# But first, let's check if there is a 3rd gender and/or a prefer not to say option
df = pd.get_dummies(df, columns=['Geography'], drop_first = True)

print(df['Gender'].nunique())

2


In [21]:
#let's also verify the columns and drop the unnecessary columns

df = df.drop(columns=['CustomerId','Surname'])
df.columns

Index(['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'Num Of Products',
       'Has Credit Card', 'Is Active Member', 'Estimated Salary', 'Churn',
       'Geography_Germany', 'Geography_Spain'],
      dtype='object')

In [22]:
# Odd. Why is there no geography france? Is France the only one in the whole dataset?  Oh right, we dropped first in one hot encoding! Not an issue.
#okay, we only have 2 values so we can map the objects to binary
#but first, let's check what they're names (M/F etc)
df['Gender'].unique()

array(['Female', 'Male'], dtype=object)

In [23]:
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
# use df['Gender' ] and not df for assignment or all your data will be lost ahaha. Don't do it.

In [24]:
#Done with preprocessing. Moving on to Splitting X and y

X = df.drop(columns=['Churn'])
y = df['Churn']

In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    test_size = 0.2,
    random_state=42
)

In [27]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_Scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [34]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, class_weight='balanced')

model.fit(X_train_Scaled,y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [35]:
from sklearn.metrics import confusion_matrix,classification_report
y_pred = model.predict(X_test_scaled)

cm = confusion_matrix(y_test,y_pred)

print("Confusion Matrix:",cm)
print(classification_report(y_test,y_pred))

Confusion Matrix: [[1159  448]
 [ 113  280]]
              precision    recall  f1-score   support

           0       0.91      0.72      0.81      1607
           1       0.38      0.71      0.50       393

    accuracy                           0.72      2000
   macro avg       0.65      0.72      0.65      2000
weighted avg       0.81      0.72      0.75      2000



In [ ]:
# Rather low recall and high false negatives
#let's check the weights
#class weight balancing helped by reducing FN from 314 to 113 but TN has increased. This is apparently known as Precision-Recall TradeOff


In [49]:
probabilities = model.predict_proba(X_test_scaled)

churned_probabilities = probabilities[:,1]

custom_threshold =0.6

y_pred_custom = (churned_probabilities >= custom_threshold).astype(int)

cn6 = confusion_matrix(y_test, y_pred_custom)
print("Confusion matrix:",cn6)
print(classification_report(y_test, y_pred_custom))

Confusion matrix: [[1338  269]
 [ 167  226]]
              precision    recall  f1-score   support

           0       0.89      0.83      0.86      1607
           1       0.46      0.58      0.51       393

    accuracy                           0.78      2000
   macro avg       0.67      0.70      0.68      2000
weighted avg       0.80      0.78      0.79      2000

